# Part 3

### Config & schema creation

In [0]:
dbutils.widgets.text("catalog", "de_assessment_dev")
CATALOG = dbutils.widgets.get("catalog")

spark.sql(f"USE CATALOG {CATALOG}")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
print("Ready")


### Episodes per season



In [0]:
episodes_per_season = (
    spark.table(f"{CATALOG}.silver.silver_episodes")
    .groupBy("show_id", "season")
    .agg(F.count("episode_id").alias("episode_count"),
         F.avg("runtime").alias("avg_runtime_mins"))
    .withColumn("rank_in_show",
        F.rank().over(Window.partitionBy("show_id").orderBy(F.desc("episode_count"))))
)

episodes_per_season.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.gold.episodes_per_season")

print("episodes_per_season:", episodes_per_season.count(), "rows")

### Avg runtime per show

In [0]:
avg_runtime_per_show = (
    spark.table(f"{CATALOG}.silver.silver_episodes")
    .join(spark.table(f"{CATALOG}.silver.silver_shows").select("show_id", "show_name").dropDuplicates(["show_id"]),
          "show_id", "left")
    .groupBy("show_id", "show_name")
    .agg(F.round(F.avg("runtime"), 1).alias("avg_runtime_mins"),
         F.count("episode_id").alias("total_episodes"))
    .withColumn("runtime_rank",
        F.rank().over(Window.orderBy(F.desc("avg_runtime_mins"))))
)

avg_runtime_per_show.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.gold.avg_runtime_per_show")

avg_runtime_per_show.orderBy("runtime_rank").show(10, truncate=False)

### Top cast members

In [0]:
top_cast = (
    spark.table(f"{CATALOG}.silver.silver_cast")
    .groupBy("cast_name")
    .agg(F.countDistinct("show_id").alias("shows_appeared_in"))
    .withColumn("cast_rank",
        F.rank().over(Window.orderBy(F.desc("shows_appeared_in"))))
    .filter(F.col("cast_rank") <= 20)
)

top_cast.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.gold.top_cast")

top_cast.orderBy("cast_rank").show(10, truncate=False)

### Most common popular genres

In [0]:
genre_popularity = (
    spark.table(f"{CATALOG}.silver.silver_shows")
    .filter(F.col("genre") != "Unknown")
    .groupBy("genre")
    .agg(F.countDistinct("show_id").alias("show_count"))
    .withColumn("genre_rank",
        F.rank().over(Window.orderBy(F.desc("show_count"))))
)

genre_popularity.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.gold.genre_popularity")

genre_popularity.orderBy("genre_rank").show(truncate=False)

In [0]:
for tbl in ["episodes_per_season", "avg_runtime_per_show", "top_cast", "genre_popularity"]:
    try:
        count = spark.table(f"{CATALOG}.gold.{tbl}").count()
        print(f"{tbl}: {count} rows")
    except Exception as e:
        print(f"Warning: could not read {tbl}: {e}")

### Databricks SQL query

In [0]:
%sql
SELECT * FROM de_assessment_dev.gold.genre_popularity ORDER BY genre_rank;

In [0]:
%sql
SELECT * FROM de_assessment_dev.gold.top_cast ORDER BY cast_rank LIMIT 10;

In [0]:
import json

gold_tables = {
    "episodes_per_season":  {"show_id", "season", "episode_count", "avg_runtime_mins"},
    "avg_runtime_per_show": {"show_id", "show_name", "avg_runtime_mins", "total_episodes"},
    "top_cast":             {"cast_name", "shows_appeared_in", "cast_rank"},
    "genre_popularity":     {"genre", "show_count", "genre_rank"},
}

counts = {}
issues = []
for tbl, expected_cols in gold_tables.items():
    try:
        df = spark.table(f"de_assessment_dev.gold.{tbl}")
        actual = set(df.columns)
        missing = expected_cols - actual
        if missing:
            issues.append(f"{tbl} missing columns: {missing}")
        cnt = df.count()
        counts[tbl] = cnt
        if cnt == 0:
            issues.append(f"{tbl} is empty")
    except Exception as e:
        issues.append(f"{tbl}: could not read table — {e}")
        counts[tbl] = -1

if issues:
    msg = "; ".join(issues)
    print(f"Gold validation FAILED: {msg}")
    dbutils.notebook.exit(json.dumps({"status": "FAILED", "reason": msg, "counts": counts}))

print("Gold validation passed:", counts)
dbutils.notebook.exit(json.dumps({"status": "OK", "counts": counts}))

## Gold Layer Business Value

| Table | Business Value |
|---|---|
| `episodes_per_season` | Helps content teams analyse season length trends and plan episode budgets |
| `avg_runtime_per_show` | Supports broadcast scheduling and advertising slot planning |
| `top_cast` | Identifies high-demand talent for investment and casting decisions |
| `genre_popularity` | Guides content commissioning strategy based on genre demand |